# Notebook 07: Revision Analyses

**Purpose:** Address reviewer comments from Bioinformatics major revision (BIOINF-2026-0189).

This notebook implements:
1. **W5** — Subtype-stratified performance analysis + temporal holdout validation
2. **W6** — Multi-PLM comparison (ESM-2 vs ESM C 600M vs ESM-1v)

## Setup

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Add project root to path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_processing import (
    PI_DRUGS, NRTI_DRUGS, NNRTI_DRUGS, ALL_DRUGS,
    get_drug_class, load_fasta
)
from src.feature_engineering import (
    HIV_PROTEASE_REFERENCE, HIV_RT_REFERENCE,
    load_esm2_model, batch_extract_pooled_embeddings
)
from src.models import per_drug_training, aggregate_drug_results
from src.evaluation import (
    delong_test, bootstrap_auc, compare_esm2_vs_baseline
)

# Set up paths
# Raw HIVDB files are in 06 HIV-ESM-2/ (two levels up from repo root)
DATA_DIR = PROJECT_ROOT.parent.parent
RESULTS_DIR = PROJECT_ROOT / 'results' / 'revision'
FIGURES_DIR = PROJECT_ROOT / 'figures' / 'revision'
EMBEDDINGS_DIR = PROJECT_ROOT / 'data' / 'embeddings'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# Device
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}')
print(f'Data dir: {DATA_DIR}')
print(f'Results dir: {RESULTS_DIR}')

# Verify data files exist
for f in ['PI_dataset.txt', 'NRTI_dataset.txt', 'NNRTI_dataset.txt']:
    exists = (DATA_DIR / f).exists()
    print(f'  {f}: {"OK" if exists else "MISSING"}')

---
## Part 1: Data Loading and Sequence Reconstruction (W5)

Reconstruct full amino acid sequences from HIVDB position columns,
then assign subtypes for stratified analysis.

In [ ]:
from src.subtype_analysis import (
    reconstruct_all_datasets,
    assign_subtypes_via_sequence_similarity,
    subtype_stratified_evaluation,
    create_temporal_split,
    temporal_holdout_evaluation
)

# Reconstruct sequences from HIVDB position columns
datasets = reconstruct_all_datasets(DATA_DIR)

for dc, df in datasets.items():
    print(f'{dc}: {len(df)} sequences, example length = {len(df["sequence"].iloc[0])}')
    print(f'  Columns: {list(df.columns)}')
    print(f'  First sequence (first 40 aa): {df["sequence"].iloc[0][:40]}...')
    print()

### 1.1 Subtype Assignment

Using Hamming distance from HXB2 (subtype B) reference as a proxy.
If nucleotide sequences are available, the Sierra API provides more accurate results.

In [ ]:
# Assign subtypes for each drug class
subtype_dfs = {}
for dc, df in datasets.items():
    print(f'\n--- {dc} ---')
    st_df = assign_subtypes_via_sequence_similarity(
        df['sequence'].tolist(),
        df['SeqID'].tolist(),
        drug_class=dc
    )
    subtype_dfs[dc] = st_df
    
    print(f'Subtype distribution:')
    print(st_df['subtype'].value_counts())
    print(f'Mean distance from B: {st_df["distance_from_B"].mean():.4f}')
    print(f'Median mutations: {st_df["n_mutations"].median():.0f}')

In [ ]:
# Visualize mutation distance distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (dc, st_df) in zip(axes, subtype_dfs.items()):
    ax.hist(st_df['distance_from_B'], bins=50, edgecolor='white', alpha=0.8)
    ax.axvline(x=0.05, color='red', linestyle='--', label='B/B-divergent cutoff')
    ax.axvline(x=0.10, color='orange', linestyle='--', label='B-divergent/non-B cutoff')
    ax.set_xlabel('Hamming distance from HXB2 (subtype B)')
    ax.set_ylabel('Count')
    ax.set_title(f'{dc} sequences')
    ax.legend(fontsize=8)

plt.suptitle('Sequence Divergence from Subtype B Reference', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'subtype_distance_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### 1.2 Subtype-Stratified Performance

Run the existing ESM-2 pipeline and report AUC per subtype group.

In [ ]:
# Load existing ESM-2 embeddings (if cached) or extract fresh
esm2_embeddings = {}
for dc in ['PI', 'NRTI', 'NNRTI']:
    cache_path = EMBEDDINGS_DIR / f'esm2_{dc}_mean.npy'
    if cache_path.exists():
        print(f'Loading cached ESM-2 embeddings for {dc}')
        esm2_embeddings[dc] = np.load(cache_path)
    else:
        print(f'Extracting ESM-2 embeddings for {dc}...')
        if dc not in datasets:
            continue
        model, alphabet, batch_converter, device = load_esm2_model(device=DEVICE)
        X = batch_extract_pooled_embeddings(
            datasets[dc]['sequence'].tolist(),
            model, alphabet, batch_converter, device,
            pooling_method='mean'
        )
        esm2_embeddings[dc] = X
        np.save(cache_path, X)
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    print(f'  {dc}: shape = {esm2_embeddings[dc].shape}')

In [ ]:
# Prepare phenotype DataFrames with class2 labels
# The raw files have fold-change values; we need to binarize them
from src.data_processing import extract_resistance_labels

pheno_dfs = {}
drug_lists = {'PI': PI_DRUGS, 'NRTI': NRTI_DRUGS, 'NNRTI': NNRTI_DRUGS}

for dc, df in datasets.items():
    pheno = pd.DataFrame()
    drugs = drug_lists[dc]
    for drug in drugs:
        if drug in df.columns:
            # Convert fold-change to binary using standard thresholds
            fc = pd.to_numeric(df[drug], errors='coerce').values
            # Use HIVDB standard: class2 thresholds vary by drug
            # Simplified: resistant if FC >= biological cutoff
            # For consistency with original pipeline, use FC >= 2.5 as default
            pheno[f'{drug}_class2'] = (fc >= 2.5).astype(float)
            pheno.loc[np.isnan(fc), f'{drug}_class2'] = np.nan
    pheno_dfs[dc] = pheno
    print(f'{dc}: {len(pheno)} samples, {len(drugs)} drugs')

In [ ]:
# Run subtype-stratified evaluation
all_stratified = []

for dc in ['PI', 'NRTI', 'NNRTI']:
    if dc not in esm2_embeddings:
        continue
    
    print(f'\n--- {dc} ---')
    subtypes = subtype_dfs[dc]['subtype']
    
    strat_df = subtype_stratified_evaluation(
        esm2_embeddings[dc],
        pheno_dfs[dc],
        subtypes,
        drug_lists[dc],
        model_type='logistic'
    )
    strat_df['drug_class'] = dc
    all_stratified.append(strat_df)

stratified_results = pd.concat(all_stratified, ignore_index=True)
stratified_results.to_csv(RESULTS_DIR / 'subtype_stratified_auc.csv', index=False)
print('\nSubtype-stratified results saved.')
print(stratified_results.groupby('subtype')['auc'].agg(['mean', 'std', 'count']))

In [ ]:
# Visualize subtype-stratified performance
fig, ax = plt.subplots(figsize=(10, 6))

plot_df = stratified_results.dropna(subset=['auc'])
sns.boxplot(data=plot_df, x='subtype', y='auc', hue='drug_class', ax=ax, palette='Set2')
ax.set_xlabel('Subtype Group', fontsize=12)
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_title('ESM-2 Performance by HIV Subtype Group', fontsize=14, fontweight='bold')
ax.set_ylim([0.7, 1.0])
ax.legend(title='Drug Class')

# Add sample counts
for i, subtype in enumerate(plot_df['subtype'].unique()):
    n = plot_df[plot_df['subtype'] == subtype]['n_samples'].sum()
    ax.text(i, 0.72, f'n={n}', ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'subtype_stratified_performance.png', dpi=300, bbox_inches='tight')
plt.show()

### 1.3 Temporal Holdout Validation

Split data by SeqID ordering (proxy for chronological order) to test
generalization to newer variant sequences.

In [ ]:
# Temporal holdout evaluation
all_temporal = []

for dc in ['PI', 'NRTI', 'NNRTI']:
    if dc not in esm2_embeddings or dc not in datasets:
        continue
    
    print(f'\n--- {dc} ---')
    train_idx, test_idx = create_temporal_split(
        datasets[dc], cutoff_quantile=0.8
    )
    
    temp_df = temporal_holdout_evaluation(
        esm2_embeddings[dc],
        pheno_dfs[dc],
        drug_lists[dc],
        train_idx, test_idx,
        model_type='logistic'
    )
    temp_df['drug_class'] = dc
    all_temporal.append(temp_df)

temporal_results = pd.concat(all_temporal, ignore_index=True)
temporal_results.to_csv(RESULTS_DIR / 'temporal_holdout_auc.csv', index=False)
print(f'\nMean temporal holdout AUC: {temporal_results["auc"].mean():.4f}')
print(temporal_results[['drug', 'auc', 'n_train', 'n_test']].to_string(index=False))

---
## Part 2: Multi-PLM Comparison (W6)

Compare ESM-2 (original), ESM C 600M (successor), and ESM-1v (variant scoring)
using the same classifier pipeline for fair comparison.

In [ ]:
from src.plm_comparison import (
    load_esmc_model, extract_esmc_embeddings,
    load_esm1v_model, extract_esm1v_embeddings,
    run_plm_comparison, format_plm_comparison_table
)

In [ ]:
# Prepare sequence and phenotype dicts for the comparison pipeline
sequences_dict = {dc: datasets[dc]['sequence'].tolist() for dc in datasets}

# Run the full comparison
plm_results = run_plm_comparison(
    sequences=sequences_dict,
    phenotypes=pheno_dfs,
    drug_lists=drug_lists,
    embeddings_cache_dir=str(EMBEDDINGS_DIR),
    models_to_run=['esm2', 'esmc', 'esm1v'],
    classifier='logistic',
    n_splits=5,
    random_state=42,
    device=DEVICE
)

plm_results.to_csv(RESULTS_DIR / 'plm_comparison_results.csv', index=False)
print('\nResults saved.')

In [ ]:
# Format comparison table
comparison_table = format_plm_comparison_table(plm_results)
print('\n=== PLM Comparison: AUC by Drug ===')
print(comparison_table.to_string())

comparison_table.to_csv(RESULTS_DIR / 'plm_comparison_table.csv')

In [ ]:
# Statistical comparison: DeLong tests between PLMs
# For this, we need the per-drug predictions from each PLM
# Re-run per_drug_training to capture predictions

print('\n=== Mean AUC by PLM ===')
summary = plm_results.groupby('plm')['auc'].agg(['mean', 'std', 'min', 'max'])
summary.columns = ['Mean AUC', 'Std', 'Min', 'Max']
print(summary.round(4))

# Per drug-class means
print('\n=== Mean AUC by PLM and Drug Class ===')
class_summary = plm_results.groupby(['plm', 'drug_class'])['auc'].mean().unstack()
print(class_summary.round(4))

In [ ]:
# Visualize PLM comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Grouped bar chart by drug
ax = axes[0]
pivot = plm_results.pivot(index='drug', columns='plm', values='auc')
pivot = pivot.sort_index()
pivot.plot(kind='bar', ax=ax, width=0.8, edgecolor='white')
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_xlabel('Drug', fontsize=12)
ax.set_title('PLM Comparison: AUC by Drug', fontsize=14, fontweight='bold')
ax.set_ylim([0.8, 1.0])
ax.legend(title='PLM Backbone')
ax.tick_params(axis='x', rotation=45)

# 2. Box plot by PLM
ax = axes[1]
plm_order = ['esm2', 'esmc', 'esm1v']
plm_labels = {'esm2': 'ESM-2\n(650M, 1280d)', 'esmc': 'ESM C\n(600M, 1152d)', 'esm1v': 'ESM-1v\n(650M, 1280d)'}
plot_df = plm_results.copy()
plot_df['plm_label'] = plot_df['plm'].map(plm_labels)

sns.boxplot(data=plot_df, x='plm_label', y='auc', ax=ax, palette='Set2',
            order=[plm_labels[p] for p in plm_order])
sns.stripplot(data=plot_df, x='plm_label', y='auc', ax=ax, color='black',
              alpha=0.4, size=4, order=[plm_labels[p] for p in plm_order])
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_xlabel('PLM Backbone', fontsize=12)
ax.set_title('PLM Comparison: Distribution Across 18 Drugs', fontsize=14, fontweight='bold')
ax.set_ylim([0.8, 1.0])

# Add mean annotations
for i, plm in enumerate(plm_order):
    mean_auc = plm_results[plm_results['plm'] == plm]['auc'].mean()
    ax.text(i, 0.81, f'mean={mean_auc:.3f}', ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'plm_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap of all results
fig, ax = plt.subplots(figsize=(8, 10))

pivot_hm = plm_results.pivot(index='drug', columns='plm', values='auc')

# Add drug class for ordering
drug_order = PI_DRUGS + NRTI_DRUGS + NNRTI_DRUGS
pivot_hm = pivot_hm.reindex([d for d in drug_order if d in pivot_hm.index])

sns.heatmap(pivot_hm, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.85, vmax=1.0, ax=ax, cbar_kws={'label': 'AUC-ROC'},
            linewidths=0.5)

ax.set_xlabel('PLM Backbone', fontsize=12)
ax.set_ylabel('Drug', fontsize=12)
ax.set_title('Multi-PLM Performance Comparison', fontsize=14, fontweight='bold')

# Add drug class separators
n_pi = len([d for d in PI_DRUGS if d in pivot_hm.index])
n_nrti = len([d for d in NRTI_DRUGS if d in pivot_hm.index])
ax.axhline(y=n_pi, color='black', linewidth=2)
ax.axhline(y=n_pi + n_nrti, color='black', linewidth=2)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'plm_comparison_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Part 3: Summary Statistics for Manuscript

Compile the key numbers needed for the revised manuscript and response letter.

In [ ]:
print('='*60)
print('REVISION SUMMARY STATISTICS')
print('='*60)

# W5: Subtype analysis
print('\n--- W5: Subtype Analysis ---')
for dc in subtype_dfs:
    st = subtype_dfs[dc]
    print(f'\n{dc}:')
    for subtype, count in st['subtype'].value_counts().items():
        pct = count / len(st) * 100
        print(f'  {subtype}: {count} ({pct:.1f}%)')

print('\n--- W5: Subtype-stratified AUC ---')
if len(all_stratified) > 0:
    for subtype in stratified_results['subtype'].unique():
        sub = stratified_results[stratified_results['subtype'] == subtype]
        print(f'{subtype}: mean AUC = {sub["auc"].mean():.4f} (n_drugs with data = {len(sub)})')

print('\n--- W5: Temporal Holdout ---')
if len(all_temporal) > 0:
    print(f'Mean AUC on temporal holdout: {temporal_results["auc"].mean():.4f}')
    print(f'Median AUC: {temporal_results["auc"].median():.4f}')
    print(f'Range: [{temporal_results["auc"].min():.4f}, {temporal_results["auc"].max():.4f}]')

# W6: PLM comparison
print('\n--- W6: PLM Comparison ---')
for plm in ['esm2', 'esmc', 'esm1v']:
    sub = plm_results[plm_results['plm'] == plm]
    print(f'{plm}: mean AUC = {sub["auc"].mean():.4f} ± {sub["auc"].std():.4f}')

print('\n--- Drugs where ESM C outperforms ESM-2 ---')
if 'esm2' in plm_results['plm'].values and 'esmc' in plm_results['plm'].values:
    esm2_aucs = plm_results[plm_results['plm'] == 'esm2'].set_index('drug')['auc']
    esmc_aucs = plm_results[plm_results['plm'] == 'esmc'].set_index('drug')['auc']
    common = esm2_aucs.index.intersection(esmc_aucs.index)
    for drug in common:
        diff = esmc_aucs[drug] - esm2_aucs[drug]
        marker = '+' if diff > 0 else ''
        print(f'  {drug}: ESM-2={esm2_aucs[drug]:.4f}, ESM C={esmc_aucs[drug]:.4f} ({marker}{diff:.4f})')

In [ ]:
# Save all results to a single Excel file for easy reference
with pd.ExcelWriter(RESULTS_DIR / 'revision_results.xlsx') as writer:
    if len(all_stratified) > 0:
        stratified_results.to_excel(writer, sheet_name='Subtype_Stratified', index=False)
    if len(all_temporal) > 0:
        temporal_results.to_excel(writer, sheet_name='Temporal_Holdout', index=False)
    plm_results.to_excel(writer, sheet_name='PLM_Comparison', index=False)
    comparison_table.to_excel(writer, sheet_name='PLM_Table')
    
    # Subtype distributions
    all_subtypes = pd.concat([
        st_df.assign(drug_class=dc) for dc, st_df in subtype_dfs.items()
    ])
    all_subtypes.to_excel(writer, sheet_name='Subtype_Assignments', index=False)

print('All revision results saved to:', RESULTS_DIR / 'revision_results.xlsx')

---
## Notes for Manuscript Revision

### W5 Response Points:
- Subtype distribution is predominantly B (expected given HIVDB composition)
- Subtype-stratified AUC shows [fill in results]
- Temporal holdout validates generalization to newer variants
- No independent genotype-phenotype dataset exists publicly (Stanford HIVDB is the standard benchmark)

### W6 Response Points:
- ESM C 600M is Meta/EvolutionaryScale's recommended successor to ESM-2 for embedding tasks
- ESM-3 is a generative model not designed for representation extraction; ESM C is the appropriate comparison
- ESM-1v provides complementary variant-effect scoring
- ESMDisPred (disorder prediction) and PepGraphormer (antimicrobial peptides) are task-specific tools, not general PLMs — inapplicable to drug resistance prediction
- InstructPLM-mu: weights not yet publicly released as of April 2026